# 00 — Přizpůsobení zadání

**Předmět:** Zpracování informací a znalostí  
**Téma:** Predikce předčasného ukončení studia (Student Dropout Prediction)  
**Dataset:** [Student Dropout Prediction Dataset — Kaggle](https://www.kaggle.com/datasets/meharshanali/student-dropout-prediction-dataset)

---

Tento notebook formálně popisuje přizpůsobení semestrálního zadání konkrétním podmínkám projektu.
Všechny volby jsou zde zdůvodněny; technická implementace je rozpracována v navazujících noteboocích.

## 1. Byznys kontext

Vysoká škola čelí problému předčasného odchodu studentů (dropout). Každý odchod představuje:

- **Ztrátu školného** — škola přichází o příjmy od studenta, který nedokončí studium.
- **Reputační škodu** — nízká míra dokončení studia zhoršuje hodnocení školy.
- **Promarněnou investici** — škola vložila prostředky do výuky, ubytování a administrativy.

**Cíl:** Vytvořit prediktivní model, který včas identifikuje studenty ohrožené odchodem, aby jim škola mohla nabídnout cílenou podporu — individuální poradenství, úpravu studijní zátěže, stipendium nebo jiný intervenční program.

> Klíčový předpoklad: **cena intervence** (poradenství, stipendium) je výrazně nižší než **cena promarněného roku školného**, proto se každá zachycená hrozba vyplácí — i za cenu části falešných poplachů.

## 2. Přizpůsobení zadání

| Parametr | Hodnota | Zdůvodnění |
|---|---|---|
| **Cílový atribut** | `Dropout` (1 = odchod, 0 = dostuduje) | Binární klasifikační úloha; přímo odpovídá byznysovému cíli |
| **Vybraná instance** | Student č. 17 z testovací množiny | Nejvyšší P(Dropout) v testovací sadě (85,9 %); typický příklad rizikového studenta |
| **Atribut zájmu** | `Study_Hours_per_Day` | Druhý nejvlivnější příznak dle SHAP; škola ho může přímo ovlivnit (studijní plán, konzultace) |
| **Podmnožina pro shlukování** | Studenti 1. semestru (`Semester == 1`) | Nejkritičtější fáze — studenti teprve adaptují na VŠ prostředí; intervence v tomto bodě má největší dopad |
| **Matice nákladů** | viz sekce 3 | Asymetrická — FN je 10× dražší než FP; odráží skutečné finanční náklady školy |
| **Klasifikační práh** | 0,35 (výchozí 0,50) | Optimalizováno na maximalizaci čistého finančního přínosu; viz notebook `05` |

## 3. Popis datasetu

Dataset pochází z platformy Kaggle a obsahuje záznamy o 10 000 studentech vysoké školy.

### Klíčové atributy

| Atribut | Typ | Popis |
|---|---|---|
| `CGPA` | numerický | Kumulativní průměr GPA (celkový studijní výsledek) |
| `Study_Hours_per_Day` | numerický | Průměrný počet hodin studia denně |
| `Attendance_Rate` | numerický | Míra docházky (%) |
| `Stress_Index` | numerický | Index stresu (1–10) |
| `Family_Income` | numerický | Příjem rodiny (s chybějícími hodnotami) |
| `Scholarship` | binární | Má student stipendium? |
| `Part_Time_Job` | binární | Pracuje student na částečný úvazek? |
| `Department` | kategorický | Studijní obor (Arts, Business, CS, Engineering, Science) |
| `Semester` | ordinální | Ročník studia (Year 1–4) |
| `Dropout` | binární | **Cílový atribut** — 1 = student odešel, 0 = dostudoval |

### Základní statistiky

- **Celkem záznamů:** 10 000
- **Výskyt Dropout = 1:** 2 354 (23,5 %)
- **Chybějící hodnoty:** `Family_Income` (~12 %), `Study_Hours_per_Day` (~8 %), `Stress_Index` (~5 %)
- **Rozdělení:** 80 % trénovací, 20 % testovací množina (stratifikováno)

## 4. Matice nákladů

Byznysový kontext je silně asymetrický: nepodchycení rizikového studenta (FN) je výrazně dražší než zbytečná intervence pro bezpečného studenta (FP).

| | **Predikce: Dropout (1)** | **Predikce: Dostuduje (0)** |
|---|---|---|
| **Skutečnost: Dropout (1)** | TP = **+45 000 Kč** | FN = **−50 000 Kč** |
| **Skutečnost: Dostuduje (0)** | FP = **−5 000 Kč** | TN = **0 Kč** |

### Interpretace

- **TP (+45 000 Kč):** Škola včas identifikuje studenta a intervenuje. Ušetří ~50 000 Kč školného, minus 5 000 Kč náklady na intervenci → čistý zisk 45 000 Kč.
- **FN (−50 000 Kč):** Rizikový student není zachycen, odejde. Škola přichází o celý rok školného (50 000 Kč).
- **FP (−5 000 Kč):** Student je označen jako rizikový, ale dostudoval by sám. Škola vynaložila 5 000 Kč na zbytečnou intervenci.
- **TN (0 Kč):** Bezpečný student správně identifikován — žádné náklady.

### Matematické zdůvodnění prahu

Intervence se vyplatí, pokud: `P(Dropout) × 50 000 > 5 000`, tj. práh = 5 000 / 50 000 = **10 %**.
V praxi zvolíme práh **0,35** jako kompromis mezi matematickým optimem a operační realizovatelností (příliš nízký práh by zahltil poradenské kapacity školy).

## 5. Vybraná instance — Student č. 17

Pro lokální vysvětlitelnost (SHAP local, LIME, Decision Tree) byl vybrán **student č. 17** z testovací množiny — student s nejvyšší pravděpodobností odchodu v celé testovací sadě.

### Klíčové charakteristiky studenta č. 17

| Atribut | Hodnota | Interpretace |
|---|---|---|
| `CGPA` | −0,91 (škálováno) | Velmi nízký studijní průměr |
| `Study_Hours_per_Day` | −1,23 (škálováno) | Studuje výrazně méně než průměr |
| `Attendance_Rate` | 52 % | Nízká docházka |
| `Scholarship` | 0 (ne) | Nemá stipendium |
| `Part_Time_Job` | 1 (ano) | Pracuje popři studiu |

**P(Dropout) = 85,9 %** — model identifikuje studenta jako vysoce rizikového.

### Decision Tree pravidlo

```
CGPA ≤ −0,223
  └── CGPA ≤ −0,838
        └── Scholarship ≤ 0,5  →  Dropout (84 % pravděpodobnost, 176 vzorků)
```

Student splňuje všechny tři podmínky → Decision Tree ho klasifikuje jako Dropout.

## 6. Atribut zájmu — `Study_Hours_per_Day`

Pro ICE analýzu (Individual Conditional Expectation) byl zvolen příznak `Study_Hours_per_Day`.

### Zdůvodnění volby

1. **Vliv na predikci:** Druhý nejvlivnější příznak dle SHAP globální analýzy (po CGPA).
2. **Ovlivnitelnost:** Na rozdíl od CGPA (historická veličina) nebo rodinného příjmu jde o chování, které škola může aktivně ovlivnit — studijní poradenství, skupiny, plány.
3. **Interpretovatelnost:** Počet hodin studia denně je intuitivní veličina pro pedagogy i studenty.

### Co ICE analýza ukáže

ICE křivka pro studenta č. 17 zobrazí, jak by se jeho P(Dropout) měnila, kdyby studoval 0–10 hodin denně — při zachování všech ostatních atributů. To škole ukáže, při jakém minimálním počtu hodin studia se predikce obrátí z Dropout na Dostuduje.

## 7. Podmnožina pro shlukování — studenti 1. semestru

Shlukování bylo provedeno na podmnožině studentů v prvním semestru (`Semester == 1`).

### Zdůvodnění

- **Kritická fáze:** První semestr je nejrizikovější — studenti se adaptují na nové prostředí, mnozí přicházejí z různých středních škol s různou úrovní přípravy.
- **Homogenita:** Studenti 1. semestru mají srovnatelný studijní kontext (stejný počet absolvovaných semestrů), což zlepšuje smysluplnost shluků.
- **Velikost:** 2 455 studentů (24,6 % datasetu) — dostatečná velikost pro K-Means.

### Příznaky pro shlukování

Pro shlukování byly použity čtyři příznaky, které nejlépe charakterizují studijní profil:

| Příznak | Zdůvodnění |
|---|---|
| `Stress_Index` | Psychická zátěž — klíčový prediktor rizika odchodu |
| `Study_Hours_per_Day` | Studijní návyky — ovlivnitelný faktor |
| `Family_Income` | Socioekonomické zázemí — determinuje dostupnost podpory |
| `CGPA` | Studijní výsledky — nejsilnější prediktor v modelu |

### Výsledky

- **K-Means K=3:** Silhouette = 0,242
- **Agglomerative (Ward):** Silhouette = 0,157, ARI = 0,326
- Podrobná analýza profilů shluků v notebocích `04a` a `04b`.

## 8. Struktura ML pipeline

```
00-intro.ipynb              ← tento notebook
01-eda.ipynb                ← průzkumná analýza dat
02-data-preprocessing.ipynb ← předzpracování, HolmanImputer, sklearn Pipeline
03a-dummy-baseline.ipynb    ← referenční model (DummyClassifier)
03b-baseline-models.ipynb   ← LR, RF, GB, DT (cross-validation)
03c-baseline-evaluation.ipynb ← metriky, feature importance
03d-hyperparameter-tuning.ipynb ← RandomizedSearchCV (50 iterací, CV=5)
04a-clustering-model.ipynb  ← K-Means + Agglomerative (Semester==1)
04b-clustering-evaluation.ipynb ← PCA, profily shluků
05-model-evaluation.ipynb   ← matice nákladů, práh, finální srovnání
06a-xai-setup.ipynb         ← příprava XAI kontextu
06b-shap-n-ice-global.ipynb ← SHAP globální + ICE
06c-shap-local.ipynb        ← SHAP lokální (student č. 17 vs. bezpečný)
06d-lime-explanations.ipynb ← LIME
06e-xai-cross-comparison.ipynb ← křížové srovnání XAI metod
06f-decision-tree-explanation.ipynb ← DT vizualizace + predikce instance
```

In [ ]:
import pandas as pd
from pathlib import Path

df = pd.read_csv(Path('../data/student_dropout_dataset_v3.csv'))

print(f'Celkem záznamů:  {len(df):,}')
print(f'Atributů:        {df.shape[1]}')
print(f'Dropout = 1:     {df["Dropout"].sum():,}  ({df["Dropout"].mean():.1%})')
print(f'Dropout = 0:     {(df["Dropout"]==0).sum():,}  ({(df["Dropout"]==0).mean():.1%})')
print()
print('Chybějící hodnoty:')
missing = df.isnull().sum()
print(missing[missing > 0].to_string())
print()
print('Typy atributů:')
print(df.dtypes.value_counts().to_string())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Matice nákladů — vizualizace
cost_matrix = np.array([[45000, -50000], [-5000, 0]])
labels = [['TP\n+45 000 Kč', 'FN\n−50 000 Kč'], ['FP\n−5 000 Kč', 'TN\n0 Kč']]

fig, ax = plt.subplots(figsize=(6, 4))
colors = np.where(cost_matrix > 0, 0.8, np.where(cost_matrix < 0, 0.2, 0.5))
im = ax.imshow(colors, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')

for i in range(2):
    for j in range(2):
        ax.text(j, i, labels[i][j], ha='center', va='center',
                fontsize=13, fontweight='bold',
                color='white' if abs(cost_matrix[i,j]) > 20000 else '#212529')

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Predikce: Dropout', 'Predikce: Dostuduje'], fontsize=11)
ax.set_yticklabels(['Skutečnost: Dropout', 'Skutečnost: Dostuduje'], fontsize=11)
ax.set_title('Matice nákladů', fontsize=14, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()
print('Matematický optimální práh: FP/(FP+TP) = 5 000/50 000 = 10 %')
print('Provozní práh zvolený v projektu: 35 %')